# Task 1: Data Science Fundamentals Assessment
**Intern:** Tejbir Singh  
**Role:** Data Analyst Intern (Hyderabad)  
**Topics Covered:** Python Basics, Data Types, Statistical Foundations, and Data Cleaning Best Practices.

In [ ]:
"""
Data Science Fundamentals Assessment - Task 1
Author: Tejbir Singh
Internship: Data Analyst Internship in Hyderabad
Description:
    Demonstrates core data science fundamentals including:
    1. Python Data Structures & Control Flow
    2. Data Types Classification (Quantitative vs. Qualitative)
    3. Statistical Foundations (Central Tendency, Dispersion, Skewness, Z-Scores)
    4. Data Preprocessing & Cleaning Best Practices (Missing values, Outliers, Scaling)
"""

import numpy as np
import pandas as pd
from scipy import stats


def demonstrate_python_basics():
    """Demonstrates core Python data structures and comprehensions."""
    print("=== 1. PYTHON BASICS & DATA STRUCTURES ===")
    
    # List, Tuple, Set, Dictionary
    metrics_list = ["Mean", "Median", "Mode", "Standard Deviation", "Variance"]
    immutable_bounds = (0.0, 100.0)
    unique_categories = set(["Retail", "Healthcare", "Finance", "Retail"])
    
    # Dictionary comprehension mapping metrics to descriptions
    metric_lengths = {metric: len(metric) for metric in metrics_list}
    
    print(f"Metrics List: {metrics_list}")
    print(f"Immutable Bounds (Tuple): {immutable_bounds}")
    print(f"Unique Categories (Set): {unique_categories}")
    print(f"Dictionary Comprehension: {metric_lengths}\n")


def generate_sample_dataset():
    """Creates a realistic synthetic dataset containing mixed data types and anomalies."""
    np.random.seed(42)
    n_records = 20
    
    data = {
        "Employee_ID": [f"EMP-{100 + i}" for i in range(n_records)],        # Nominal
        "Department": np.random.choice(["Analytics", "Engineering", "Sales"], n_records), # Nominal
        "Experience_Level": np.random.choice(["Junior", "Mid", "Senior"], n_records),     # Ordinal
        "Age": np.random.randint(22, 55, n_records).astype(float),          # Discrete / Ratio
        "Annual_Salary_USD": np.round(np.random.normal(75000, 15000, n_records), 2),      # Continuous / Ratio
        "Performance_Score": np.round(np.random.uniform(2.5, 5.0, n_records), 2)          # Continuous / Interval
    }
    
    df = pd.DataFrame(data)
    
    # Introduce realistic data quality issues (missing values & an outlier)
    df.loc[2, "Age"] = np.nan
    df.loc[7, "Annual_Salary_USD"] = np.nan
    df.loc[15, "Annual_Salary_USD"] = 285000.00  # Extreme outlier
    
    return df


def statistical_foundations(df):
    """Computes descriptive statistics, dispersion, and distribution metrics."""
    print("=== 2. STATISTICAL FOUNDATIONS ===")
    salary = df["Annual_Salary_USD"].dropna()
    
    mean_val = salary.mean()
    median_val = salary.median()
    mode_val = salary.mode().iloc[0]
    std_val = salary.std(ddof=1)
    var_val = salary.var(ddof=1)
    q1 = salary.quantile(0.25)
    q3 = salary.quantile(0.75)
    iqr = q3 - q1
    skewness = stats.skew(salary)
    
    print(f"Count (Valid): {len(salary)}")
    print(f"Mean Salary:   ${mean_val:,.2f}")
    print(f"Median Salary: ${median_val:,.2f}")
    print(f"Std Deviation: ${std_val:,.2f}")
    print(f"Variance:      {var_val:,.2f}")
    print(f"IQR (Q3 - Q1): ${iqr:,.2f} (Q1=${q1:,.2f}, Q3=${q3:,.2f})")
    print(f"Skewness:      {skewness:.3f} (Positive skew due to high salary outlier)\n")
    
    return q1, q3, iqr


def data_cleaning_best_practices(df, q1, q3, iqr):
    """Applies reproducible data cleaning: imputation, outlier capping, and encoding."""
    print("=== 3. DATA CLEANING & INDUSTRY BEST PRACTICES ===")
    df_clean = df.copy()
    
    # 1. Missing Value Imputation (Median for skewed distributions, Mean for symmetric)
    age_median = df_clean["Age"].median()
    salary_median = df_clean["Annual_Salary_USD"].median()
    
    df_clean["Age"] = df_clean["Age"].fillna(age_median)
    df_clean["Annual_Salary_USD"] = df_clean["Annual_Salary_USD"].fillna(salary_median)
    print(f"Imputed missing Age with median ({age_median}) and Salary with median (${salary_median:,.2f}).")
    
    # 2. Outlier Detection & Winsorization (IQR Method)
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df_clean[(df_clean["Annual_Salary_USD"] < lower_bound) | (df_clean["Annual_Salary_USD"] > upper_bound)]
    print(f"Detected {len(outliers)} salary outlier(s) above upper bound (${upper_bound:,.2f}).")
    
    df_clean["Annual_Salary_Capped"] = np.clip(df_clean["Annual_Salary_USD"], lower_bound, upper_bound)
    
    # 3. Ordinal Encoding & Z-Score Standardization
    exp_mapping = {"Junior": 1, "Mid": 2, "Senior": 3}
    df_clean["Experience_Encoded"] = df_clean["Experience_Level"].map(exp_mapping)
    
    sal_mean = df_clean["Annual_Salary_Capped"].mean()
    sal_std = df_clean["Annual_Salary_Capped"].std()
    df_clean["Salary_ZScore"] = (df_clean["Annual_Salary_Capped"] - sal_mean) / sal_std
    
    print("\nCleaned Dataset Preview (First 5 Rows):")
    print(df_clean[["Employee_ID", "Department", "Experience_Encoded", "Age", "Annual_Salary_Capped", "Salary_ZScore"]].head())
    return df_clean


if __name__ == "__main__":
    demonstrate_python_basics()
    raw_df = generate_sample_dataset()
    q1, q3, iqr = statistical_foundations(raw_df)
    cleaned_df = data_cleaning_best_practices(raw_df, q1, q3, iqr)
